# Unit Testing with unittest and pytest

This notebook provides a comprehensive introduction to unit testing in Python using the built-in `unittest` framework and the popular `pytest` library.

## 1. Import Required Libraries

Let's start by installing and importing the libraries we need for this tutorial.

In [ ]:
# Install required packages
!pip install pytest pytest-cov mock

# Import standard testing modules
import unittest
import pytest

# For mocking
from unittest import mock
from unittest.mock import Mock, MagicMock, patch

# For coverage reporting
import coverage

# Some standard libraries we'll use in our examples
import math
import os
import sys
import json
import tempfile
from datetime import datetime

## 2. Unit Testing Concepts

Unit testing is a software testing methodology where individual units or components of software are tested. The purpose is to validate that each unit performs as designed.

### Key Concepts in Unit Testing:

1. **Test Case**: A set of conditions to determine whether a system works correctly
2. **Test Suite**: A collection of test cases
3. **Test Fixture**: The fixed state used as a baseline for running tests
4. **Test Runner**: A component that orchestrates test execution
5. **Assertion**: A statement that checks if a condition is true

### Benefits of Unit Testing:

- **Early Bug Detection**: Identify issues early in development
- **Documentation**: Tests serve as documentation for how code should behave
- **Refactoring Safety**: Change code with confidence
- **Design Improvement**: Leads to better code organization
- **Integration Simplicity**: Easier to integrate components that have been tested

Let's create a simple function to demonstrate unit testing:

In [ ]:
# A function to calculate the area of a circle
def calculate_circle_area(radius):
    """
    Calculate the area of a circle given its radius.
    
    Args:
        radius (float): Radius of the circle
        
    Returns:
        float: Area of the circle
        
    Raises:
        ValueError: If radius is negative
    """
    if radius < 0:
        raise ValueError("Radius cannot be negative")
    return math.pi * (radius ** 2)

# A simple calculator class for demonstration
class Calculator:
    def add(self, a, b):
        return a + b
    
    def subtract(self, a, b):
        return a - b
    
    def multiply(self, a, b):
        return a * b
    
    def divide(self, a, b):
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

## 3. Testing with the unittest Module

The `unittest` module is Python's built-in testing framework. It supports test automation, sharing of setup and shutdown code, and independence of tests from the reporting framework.

### Key Components of unittest:

1. **TestCase**: Base class for test cases
2. **TestSuite**: Collection of test cases
3. **TestRunner**: Executes tests and collects results

### Common Assertions in unittest:

- `assertEqual(a, b)`: Verify that a == b
- `assertNotEqual(a, b)`: Verify that a != b
- `assertTrue(x)`: Verify that x is True
- `assertFalse(x)`: Verify that x is False
- `assertRaises(exc, fun, *args)`: Verify that fun(*args) raises exc
- `assertAlmostEqual(a, b)`: Verify that round(a-b, 7) == 0

## 4. Creating Test Cases with unittest

Let's create a test case for our circle area function using unittest:

In [ ]:
class TestCircleArea(unittest.TestCase):
    def test_area(self):
        # Test areas for different radii
        self.assertAlmostEqual(calculate_circle_area(1), math.pi)
        self.assertAlmostEqual(calculate_circle_area(0), 0)
        self.assertAlmostEqual(calculate_circle_area(2.5), math.pi * 6.25)
    
    def test_values(self):
        # Test that function raises the appropriate error for invalid values
        self.assertRaises(ValueError, calculate_circle_area, -1)
        self.assertRaises(ValueError, calculate_circle_area, -10)
        
    def test_types(self):
        # Test with different input types
        self.assertAlmostEqual(calculate_circle_area(1.0), math.pi)
        self.assertAlmostEqual(calculate_circle_area(1), math.pi)
        
        # Test with a type conversion edge case
        with self.assertRaises(TypeError):
            calculate_circle_area("2")

In [ ]:
# Let's also create a test case for our Calculator class
class TestCalculator(unittest.TestCase):
    def setUp(self):
        # This method runs before each test
        self.calc = Calculator()
    
    def test_add(self):
        self.assertEqual(self.calc.add(3, 5), 8)
        self.assertEqual(self.calc.add(-1, 1), 0)
    
    def test_subtract(self):
        self.assertEqual(self.calc.subtract(5, 3), 2)
        self.assertEqual(self.calc.subtract(1, 1), 0)
    
    def test_multiply(self):
        self.assertEqual(self.calc.multiply(3, 5), 15)
        self.assertEqual(self.calc.multiply(-1, 1), -1)
    
    def test_divide(self):
        self.assertEqual(self.calc.divide(6, 3), 2)
        self.assertEqual(self.calc.divide(1, 1), 1)
        
        # Test division by zero
        with self.assertRaises(ValueError):
            self.calc.divide(1, 0)

## 5. Running Tests with unittest

The unittest module provides several ways to run tests. In this notebook, we'll use the `unittest.main()` function.

In [ ]:
# Running tests in a Jupyter Notebook
unittest.main(argv=['first-arg-is-ignored'], exit=False)

In a typical Python script, you would run tests using:

```python
if __name__ == '__main__':
    unittest.main()
```

Or from the command line:

```bash
python -m unittest test_module.py
```

### Test Discovery

unittest can automatically discover and run tests:

```bash
python -m unittest discover -s tests -p "test_*.py"
```

## 6. Test Fixtures with unittest

Test fixtures are resources needed by a test. unittest provides several methods for setting up and tearing down test fixtures:

- `setUp()`: Called before each test method
- `tearDown()`: Called after each test method
- `setUpClass()`: Called once before any test methods
- `tearDownClass()`: Called once after all test methods

In [ ]:
# First, let's define a simple User class for our example
class User:
    def __init__(self, username, email):
        self.username = username
        self.email = email
        self.is_active = True
        
    def deactivate(self):
        self.is_active = False
        
    def activate(self):
        self.is_active = True

# Now let's create a UserManager class
class UserManager:
    def __init__(self):
        self.users = {}
    
    def add_user(self, username, email):
        if username in self.users:
            raise ValueError(f"User '{username}' already exists")
        user = User(username, email)
        self.users[username] = user
        return user
    
    def get_user(self, username):
        return self.users.get(username)
    
    def remove_user(self, username):
        if username in self.users:
            del self.users[username]
            return True
        return False

In [ ]:
class TestUserManager(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Called once before any tests
        print("Setting up class fixtures...")
        # For example, connect to a test database
        cls.temp_file = tempfile.NamedTemporaryFile(delete=False)
    
    @classmethod
    def tearDownClass(cls):
        # Called once after all tests
        print("Tearing down class fixtures...")
        # For example, disconnect from the test database
        os.unlink(cls.temp_file.name)
    
    def setUp(self):
        # Called before each test method
        self.manager = UserManager()
        # Add some users for testing
        self.manager.add_user("user1", "user1@example.com")
        self.manager.add_user("user2", "user2@example.com")
    
    def tearDown(self):
        # Called after each test method
        # Clean up any resources
        self.manager = None
    
    def test_add_user(self):
        user = self.manager.add_user("testuser", "test@example.com")
        self.assertEqual(user.username, "testuser")
        self.assertEqual(user.email, "test@example.com")
        self.assertTrue(user.is_active)
        
        # Test adding a duplicate user
        with self.assertRaises(ValueError):
            self.manager.add_user("user1", "duplicate@example.com")
    
    def test_get_user(self):
        user = self.manager.get_user("user1")
        self.assertIsNotNone(user)
        self.assertEqual(user.username, "user1")
        
        # Test getting non-existent user
        self.assertIsNone(self.manager.get_user("nonexistent"))

## 7. Introduction to pytest

pytest is a more modern testing framework that simplifies test writing with these advantages:

1. **Simple Syntax**: Write tests as simple functions instead of classes
2. **Powerful Fixtures**: More flexible fixture management
3. **Detailed Test Reports**: Better failure information
4. **Parameterized Testing**: Run tests with multiple inputs
5. **Plugin System**: Extend functionality with plugins

In [ ]:
# Basic pytest tests
def test_circle_area_with_pytest():
    # Test areas for different radii
    assert calculate_circle_area(1) == pytest.approx(math.pi)
    assert calculate_circle_area(0) == 0
    assert calculate_circle_area(2.5) == pytest.approx(math.pi * 6.25)

def test_circle_area_values_with_pytest():
    # Test that function raises the appropriate error for invalid values
    with pytest.raises(ValueError):
        calculate_circle_area(-1)
        
    with pytest.raises(ValueError):
        calculate_circle_area(-10)

## 8. Creating Test Files with pytest

In a real project, tests are typically organized in separate files. Here's how you would structure a pytest test file:

```python
# File: test_calculator.py
import pytest
from calculator import Calculator

def test_add():
    calc = Calculator()
    assert calc.add(3, 5) == 8
    assert calc.add(-1, 1) == 0
```

Let's create some test examples:

In [ ]:
# Define a simple string utility module
class StringUtils:
    @staticmethod
    def reverse(s):
        if not isinstance(s, str):
            raise TypeError("Input must be a string")
        return s[::-1]
    
    @staticmethod
    def is_palindrome(s):
        if not isinstance(s, str):
            raise TypeError("Input must be a string")
        
        # Remove non-alphanumeric characters and convert to lowercase
        s = ''.join(c for c in s if c.isalnum()).lower()
        return s == s[::-1]

In [ ]:
# Now let's write pytest tests for StringUtils
def test_reverse():
    assert StringUtils.reverse("hello") == "olleh"
    assert StringUtils.reverse("python") == "nohtyp"
    assert StringUtils.reverse("") == ""
    
    # Test with non-string input
    with pytest.raises(TypeError):
        StringUtils.reverse(123)

def test_is_palindrome():
    assert StringUtils.is_palindrome("racecar") == True
    assert StringUtils.is_palindrome("A man, a plan, a canal: Panama") == True
    assert StringUtils.is_palindrome("hello") == False

## 9. Running Tests with pytest

pytest offers a simple command-line interface to run tests. In a Jupyter notebook, we can use `pytest.main()`:

In [ ]:
# Running pytest in a Jupyter notebook
# Note: This will run all tests (unittest and pytest)
pytest.main(['-v'])

In a typical project, you would run pytest from the command line:

```bash
pytest                  # Run all tests in the current directory
pytest test_file.py     # Run tests in a specific file
pytest -v               # Run tests with verbose output
pytest -k "add"         # Run tests with names containing "add"
```

## 10. pytest Fixtures

pytest fixtures provide a way to set up and tear down resources needed for tests. They're more flexible than unittest's fixtures.

In [ ]:
# Define a simple database class for our example
class SimpleDB:
    def __init__(self):
        self.data = {}
        self.connected = False
    
    def connect(self):
        self.connected = True
        print("Database connected")
    
    def disconnect(self):
        self.connected = False
        print("Database disconnected")
    
    def insert(self, key, value):
        if not self.connected:
            raise RuntimeError("Database not connected")
        self.data[key] = value
    
    def get(self, key):
        if not self.connected:
            raise RuntimeError("Database not connected")
        return self.data.get(key)

In [ ]:
# Now let's write tests with fixtures
@pytest.fixture
def db():
    """Fixture that provides a SimpleDB instance."""
    print("\nSetting up database")
    db = SimpleDB()
    db.connect()
    
    # Setup is done, pass control to the test
    yield db
    
    # Teardown (executed after the test is done)
    print("Tearing down database")
    db.disconnect()

@pytest.fixture
def populated_db(db):
    """Fixture that provides a SimpleDB instance with some data."""
    # Notice how this fixture depends on the 'db' fixture
    db.insert("key1", "value1")
    db.insert("key2", "value2")
    return db

# Tests that use the fixtures
def test_db_insert(db):
    db.insert("test_key", "test_value")
    assert db.get("test_key") == "test_value"

def test_db_get(populated_db):
    assert populated_db.get("key1") == "value1"
    assert populated_db.get("key2") == "value2"
    assert populated_db.get("nonexistent") is None

### Fixture Scopes

pytest fixtures can have different scopes, which determine how often they're created:

- `function`: Created for each test function (default)
- `class`: Created once per test class
- `module`: Created once per module
- `session`: Created once per test session

In [ ]:
@pytest.fixture(scope="module")
def module_db():
    """A database fixture that lasts for the entire module."""
    print("\nSetting up module database")
    db = SimpleDB()
    db.connect()
    yield db
    print("Tearing down module database")
    db.disconnect()

## 11. Parameterized Testing with pytest

pytest allows you to run the same test with multiple sets of inputs using `@pytest.mark.parametrize`:

In [ ]:
@pytest.mark.parametrize("input_str, expected", [
    ("hello", "olleh"),
    ("python", "nohtyp"),
    ("", ""),
    ("a", "a")
])
def test_reverse_parametrized(input_str, expected):
    assert StringUtils.reverse(input_str) == expected

@pytest.mark.parametrize("a, b, expected", [
    (1, 2, 3),
    (5, 3, 8),
    (-1, 1, 0),
    (0, 0, 0)
])
def test_calculator_add_parametrized(a, b, expected):
    calc = Calculator()
    assert calc.add(a, b) == expected

## 12. Mocking in Tests

Mocking is a technique for replacing parts of your system under test with mock objects. It's useful for isolating units of code from their dependencies.

In [ ]:
# First, let's define a service that we'll need to mock
class WeatherService:
    def get_temperature(self, city):
        """In a real application, this would call an external API."""
        # This is just a placeholder - in real code, this would call an API
        return 25  # Always returns 25°C

# Now let's define a class that uses the service
class WeatherReport:
    def __init__(self, weather_service):
        self.weather_service = weather_service
    
    def format_temperature(self, city):
        temp = self.weather_service.get_temperature(city)
        return f"The temperature in {city} is {temp}°C"

In [ ]:
def test_format_temperature():
    # Create a mock for the weather service
    mock_service = MagicMock()
    mock_service.get_temperature.return_value = 15
    
    # Create a WeatherReport with the mock service
    report = WeatherReport(mock_service)
    
    # Test the format_temperature method
    result = report.format_temperature("London")
    assert result == "The temperature in London is 15°C"
    
    # Verify that get_temperature was called with "London"
    mock_service.get_temperature.assert_called_once_with("London")

In [ ]:
# Using patch as a decorator
def get_current_time():
    return datetime.now()

def format_greeting(name):
    current_time = get_current_time()
    hour = current_time.hour
    
    if 5 <= hour < 12:
        return f"Good morning, {name}!"
    elif 12 <= hour < 18:
        return f"Good afternoon, {name}!"
    else:
        return f"Good evening, {name}!"

@patch('__main__.get_current_time')
def test_format_greeting_morning(mock_get_time):
    # Configure the mock to return a morning time
    mock_time = datetime(2023, 1, 1, 8, 30)  # 8:30 AM
    mock_get_time.return_value = mock_time
    
    greeting = format_greeting("Alice")
    assert greeting == "Good morning, Alice!"

## 13. Test Coverage Measurement

Test coverage measures how much of your code is tested. pytest-cov is a plugin that provides coverage reporting:

In [ ]:
# In a real project, you would run:
# pytest --cov=mypackage tests/

# For demonstration, we'll use the coverage module directly
def demonstrate_coverage():
    cov = coverage.Coverage()
    cov.start()
    
    # Run some tests
    test_reverse()
    test_is_palindrome()
    
    cov.stop()
    cov.save()
    
    # Print report
    print("\nCoverage Report:")
    cov.report()

# Uncomment to run the coverage demonstration
# demonstrate_coverage()

## 14. Best Practices for Unit Testing

1. **Test One Thing at a Time**: Each test should verify a single behavior
2. **Keep Tests Independent**: Tests should not depend on each other
3. **Use Descriptive Test Names**: Names should describe what's being tested
4. **Arrange-Act-Assert**: Structure tests in three phases:
   - Arrange: Set up the test conditions
   - Act: Perform the action being tested
   - Assert: Verify the expected outcome
5. **Use Fixtures for Setup/Teardown**: Keep test setup and cleanup code in fixtures
6. **Test Edge Cases**: Include tests for boundary conditions and error cases
7. **Keep Tests Fast**: Unit tests should run quickly
8. **Use Mocks for External Dependencies**: Isolate units from external services
9. **Test Failure Conditions**: Make sure your code handles errors correctly
10. **Run Tests Frequently**: Ideally as part of a CI/CD pipeline

## Summary

In this notebook, we've covered:

- The fundamentals of unit testing
- How to use Python's built-in `unittest` framework
- How to use the more modern `pytest` framework
- Test fixtures and parameterized tests
- Mocking external dependencies
- Measuring test coverage
- Best practices for effective unit testing

Both `unittest` and `pytest` are powerful tools for ensuring code quality and reliability. While `unittest` comes with the standard library, `pytest` offers a more modern and flexible approach to testing.